# ArmorVault — One-click OCR-VL test

This is the simple version: no Google Drive, no uploads, and no real documents. It runs PaddleOCR-VL on CPU in a short-lived worker process, then runs MiniCPM-V on the T4 GPU after that process exits. This avoids Colab CUDA conflicts in PaddlePaddle.

Select **Runtime → Change runtime type → T4 GPU**, then choose **Run all**.

In [ ]:
import os, subprocess, sys
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
except Exception as exc:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU first.') from exc
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
%pip install -q paddlepaddle==3.3.0
%pip install -q 'paddleocr[doc-parser]>=3.7.0,<4' 'transformers>=4.51,<5' 'accelerate>=1.4' 'autoawq>=0.2.9' pillow
print('Dependencies installed.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image
import json, subprocess, sys, time

root = Path('/content/armorvault-one-click')
root.mkdir(exist_ok=True)
image_path = root / 'public_demo.png'
urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/paddleocr_vl_demo.png', image_path)
display(Image.open(image_path))

# Paddle runs in a separate process and exits before MiniCPM is loaded.
worker = r'''
import sys, time
from pathlib import Path
from paddleocr import PaddleOCRVL
input_path, output_path = Path(sys.argv[1]), Path(sys.argv[2])
pipeline = PaddleOCRVL(pipeline_version='v1.6', device='cpu', use_doc_orientation_classify=False, use_doc_unwarping=False)
parts = []
for result in pipeline.predict(str(input_path)):
    result.save_to_json(save_path=str(output_path))
    result.save_to_markdown(save_path=str(output_path))
for path in sorted(output_path.iterdir()):
    if path.suffix.lower() in {'.json', '.md'}:
        parts.append('--- ' + path.name + ' ---\n' + path.read_text(encoding='utf-8', errors='replace'))
(output_path / 'extraction.txt').write_text('\n\n'.join(parts), encoding='utf-8')
print('PaddleOCR-VL complete')
'''
worker_path = root / 'paddle_worker.py'
paddle_output = root / 'paddle-output'
paddle_output.mkdir(exist_ok=True)
worker_path.write_text(worker, encoding='utf-8')
started = time.perf_counter()
subprocess.run([sys.executable, str(worker_path), str(image_path), str(paddle_output)], check=True)
print({'paddleSeconds': round(time.perf_counter() - started, 2), 'extraction': str(paddle_output / 'extraction.txt')})

In [ ]:
import json, re, time, torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

extraction_path = root / 'paddle-output' / 'extraction.txt'
extraction = extraction_path.read_text(encoding='utf-8', errors='replace')[:60000]
model_name = 'openbmb/MiniCPM-V-4_5-AWQ'
started = time.perf_counter()
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, device_map='auto', low_cpu_mem_usage=True).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
init_seconds = time.perf_counter() - started
prompt = '''Read this public test document using the image and PaddleOCR-VL extraction. Return JSON only with exactly these keys: documentType, documentNumber, holderName, issueDate, expiryDate, totalAmount, currency, mrzLines. Use null for uncertain scalar values and [] for absent MRZ lines. Dates must be YYYY-MM-DD. Never invent unreadable characters.\n\nPaddleOCR-VL extraction:\n''' + extraction
started = time.perf_counter()
answer = model.chat(msgs=[{'role': 'user', 'content': [Image.open(image_path).convert('RGB'), prompt]}], tokenizer=tokenizer, enable_thinking=False)
inference_seconds = time.perf_counter() - started
cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', answer.strip(), flags=re.I | re.S)
try:
    fields = json.loads(cleaned)
except json.JSONDecodeError:
    start, end = cleaned.find('{'), cleaned.rfind('}')
    if start < 0 or end <= start:
        raise RuntimeError('MiniCPM did not return a JSON object.')
    fields = json.loads(cleaned[start:end + 1])
result = {'extractor': 'PaddleOCR-VL-0.9B', 'understandingModel': model_name, 'miniCPMInitializationSeconds': round(init_seconds, 2), 'miniCPMInferenceSeconds': round(inference_seconds, 2), 'fields': fields}
(root / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))

The final result is saved inside the temporary Colab session at `/content/armorvault-one-click/result.json`. No Google Drive permission is requested.